Final implementation of LDA After Filtering For Final Comparison of Results

In [1]:
import mne
import numpy as np
from numpy import matlib as mb
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import warnings
from sklearn import metrics
from collections import defaultdict
import regex as re
from sklearn.metrics import accuracy_score
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from sklearn.metrics import roc_auc_score
from models.LDA_noClassBalance import LDAClassifierNCB
from models.LDA import LDAClassifier
from src.Dataset import Dataset
from src.Speller import Speller
warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")

Divide and Separate Training and Testing Paths

In [2]:
def group_paths_by_participant(data_paths):
    participant_files = defaultdict(lambda: {'train': [], 'test': []})
    for path in data_paths:
        parts = path.split(os.sep)
        participant_id = parts[6]

        if 'Train' in path or 'train' in path:
            participant_files[participant_id]['train'].append(path)
        elif 'Test' in path or 'test' in path:
            participant_files[participant_id]['test'].append(path)
        else:
            print(f"Unclassified path: {path}")

    return participant_files


In [3]:
important_channels = ['EEG_Fz', 'EEG_Cz', 'EEG_Pz', 'EEG_P3',
                      'EEG_PO7', 'EEG_PO8', 'EEG_P4', 'EEG_Oz']

#first using default values
ds = Dataset(
    glob_path=os.path.join(project_root, "data", "*", "*", "*", "CB", "*"),
    tmin=0,
    tmax=0.8, 
)

participant_files = group_paths_by_participant(ds.data_paths)
participants = list(participant_files.keys())

Training LDA with Preprocessing

In [4]:
#Now With Filtering

studyL_results = []

# Loop through all participants
for participant_id in participant_files.keys():

    print(f"\nProcessing participant {participant_id}")

    train_files = participant_files[participant_id]['train']
    test_files  = participant_files[participant_id]['test']

    X_train, y_train = [], []
    X_test, y_test   = [], []

    # Load train data
    for path in train_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            use_car=True, 
            notch_filter=True
            
        )
        X, y = ds[0]
        X_train.append(X)
        y_train.append(y)

    # Load test data
    for path in test_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            use_car=True, 
            notch_filter=True
        )
        X, y = ds[0]
        X_test.append(X)
        y_test.append(y)

    X_train = np.vstack(X_train)
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    y_train = np.concatenate(y_train)

    X_test = np.vstack(X_test)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    y_test = np.concatenate(y_test)

    #train on train partition
    lda = LDAClassifier(solver='svd', shrinkage=None) 
    lda.fit(X_train_flat, y_train)

    #test on test partition
    scores = lda.predict_scores(X_test_flat, y_test)
    y_pred = lda.model.predict(X_test_flat)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, scores)

    print(f"Participant {participant_id} LDA Accuracy: {acc:.3f}, AUC: {auc:.3f}")

    #decode and predict characters
    character_accuracies = []
    for path in test_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8, 
            notch_filter=True, 
            use_car=True
            
        )
        X_test_specific, y_test_specific = ds[0]
        X_test_specific_flat = X_test_specific.reshape(X_test_specific.shape[0], -1)

        scores_specific = lda.model.decision_function(X_test_specific_flat)

        # speller implementation
        raw = ds.raw
        epochs = ds.epochs
        char_ch_names = [ch for ch in raw.ch_names if re.match(r'^[A-Za-z0-9]+_\d+_\d+$', ch)]
        char_ch_indices = [raw.ch_names.index(ch) for ch in char_ch_names]

        curr_target_idx = raw.ch_names.index('CurrentTarget')
        phase_idx = raw.ch_names.index('PhaseInSequence')
        data = raw.get_data()
        stim_indices = np.where(data[phase_idx] == 2)[0]
        changes = np.diff(data[curr_target_idx], prepend=data[curr_target_idx][0]-1)
        target_onsets = stim_indices[np.isin(stim_indices, np.where(changes != 0)[0])]
        target_codes = data[curr_target_idx][target_onsets].astype(int)
        current_target_events = np.array([[onset, 0, code] for onset, code in zip(target_onsets, target_codes)])

        pcr = {
            'epochs': epochs,
            'raw_data': raw,
            'character_channels': char_ch_indices,
            'current_target_events': current_target_events
        }

        speller_grid = [
            "A","B","C","D","E","F","G","H","I","J","K","L","M",
            "N","O","P","Q","R","S","T","U","V","W","X","Y","Z",
            "_","1","2","3","4","5","6","7","8","9"
        ]
        speller = Speller(speller_grid)

        predictions = speller.run(pcr, clf=None, X=scores_specific, y=1)
        metrics = speller.get_metrics()
        character_accuracies.append(metrics['accuracy'])

    avg_char_acc = np.mean(character_accuracies)
    print(f"Average character accuracy: {avg_char_acc:.3f}")
    #print("Target", metrics['target'])
    #print("Prediction", metrics['prediction'])


    # Store results
    studyL_results.append({
        "participant": participant_id,
        "LDA_accuracy": acc,
        "LDA_auc": auc,
        "characcuracy": avg_char_acc,
    })

# Summary DataFrame
results_df = pd.DataFrame(studyL_results)
print("\nStudy L Summary")
print(results_df)
print("Average AUC:", results_df["LDA_auc"].mean())
print("Average Model Accuracy:", results_df["LDA_accuracy"].mean())
print("Average Character Accuracy Across Study L:", results_df["characcuracy"].mean())


Processing participant L_01
Accuracy: 0.4778
AUC: 0.7242
Participant L_01 LDA Accuracy: 0.478, AUC: 0.724
Average character accuracy: 0.667

Processing participant L_02
Accuracy: 0.4161
AUC: 0.6808
Participant L_02 LDA Accuracy: 0.416, AUC: 0.681
Average character accuracy: 0.533

Processing participant L_03
Accuracy: 0.6944
AUC: 0.8816
Participant L_03 LDA Accuracy: 0.694, AUC: 0.882
Average character accuracy: 1.000

Processing participant L_04
Accuracy: 0.7069
AUC: 0.8703
Participant L_04 LDA Accuracy: 0.707, AUC: 0.870
Average character accuracy: 1.000

Processing participant L_05
Accuracy: 0.7500
AUC: 0.9144
Participant L_05 LDA Accuracy: 0.750, AUC: 0.914
Average character accuracy: 1.000

Processing participant L_06
Accuracy: 0.3815
AUC: 0.7051
Participant L_06 LDA Accuracy: 0.381, AUC: 0.705
Average character accuracy: 0.700

Processing participant L_07
Accuracy: 0.3553
AUC: 0.6120
Participant L_07 LDA Accuracy: 0.355, AUC: 0.612
Average character accuracy: 0.200

Processing p